<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 16px; margin-bottom: 10px;">
  <div style="text-align: center;">
    <h1 style="color: #e94560; font-size: 2.2em; margin: 0 0 8px 0; letter-spacing: 2px;">🏗️ Q-LEARNING</h1>
    <h2 style="color: #ffffff; font-size: 1.4em; margin: 0 0 16px 0; font-weight: 300;">Applied Deep Learning for Civil Engineers</h2>
    <div style="background: rgba(233,69,96,0.15); border: 1px solid #e94560; border-radius: 8px; padding: 12px 24px; display: inline-block;">
      <p style="color: #a8dadc; margin: 0; font-size: 1.05em;">📖 <b>Lecture 12</b> — Reinforcement Learning &amp; Embodied AI</p>
      <p style="color: #a8dadc; margin: 4px 0 0 0; font-size: 0.95em;">Dr. Jiaji Wang &nbsp;|&nbsp; Department of Civil Engineering &nbsp;|&nbsp; HKU</p>
    </div>
  </div>
</div>

---

## 📋 Content Overview

This content implements **three progressive levels** of Q-Learning, all grounded in a civil engineering inspection scenario:

| Section | Content | Slides |
|------|---------|--------|
| **Section 0** | Environment setup & imports | — |
| **Section 1** | Grid World environment (MDP) | Slides 13–17 |
| **Section 2** | Tabular Q-Learning + Bellman update | Slides 19–21 |
| **Section 3** | Training visualisation | Slide 31 |
| **Section 4** | Deep Q-Network (DQN) | Slides 22–31 |
| **Section 5** | DQN training visualisation | Slide 31 |

> ⚡ **Each section is self-contained** — run them in order from top to bottom, or re-run any individual cell independently.

---

### 🏛️ Civil Engineering Context

A **robot inspector** navigates a 6×6 construction floor-plan:
- 🔵 **START** — robot's initial position  
- 🟢 **GOAL** — inspection target (reward **+10**)  
- 🔴 **HAZARD** — danger zone, e.g. open pit (reward **−10**)  
- ⬜ **Free cell** — walkable floor (reward **−1** per step)

---
## ⚙️ Section 0 — Setup & Imports

<div style="background:#f0f4ff; border-left: 5px solid #4a90d9; padding: 14px 18px; border-radius: 6px;">
<b>What this cell does:</b> Imports all required libraries and checks whether PyTorch is available for Part 3 (DQN). Run this cell first before anything else.
</div>

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 0 — IMPORTS & CONFIGURATION
#  Run this cell first. All subsequent cells depend on it.
# ─────────────────────────────────────────────────────────────

import os
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque

# ── Output directory ──────────────────────────────────────────
# Change this path to wherever you want figures saved.
OUTPUT_DIR = r'D:\Q-Learning'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_path(filename):
    """Helper: returns full save path for a given filename."""
    return os.path.join(OUTPUT_DIR, filename)

# ── Reproducibility (optional) ────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── PyTorch (Part 4 / DQN only) ───────────────────────────────
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    torch.manual_seed(SEED)
    TORCH_AVAILABLE = True
    print(f'✅  PyTorch {torch.__version__} loaded — DQN (Section 4) is available.')
except ImportError:
    TORCH_AVAILABLE = False
    print('⚠️  PyTorch not found. Run:  pip install torch')
    print('    Cell 4 (DQN) will be skipped.')

print(f'✅  NumPy  {np.__version__}')
print(f'✅  Output directory: {OUTPUT_DIR}')
print('\n🚀  Setup complete — proceed to Cell 1.')

---
## 🏗️ Section 1 — Grid World Environment (MDP)

<div style="background:#fff8e1; border-left: 5px solid #f39c12; padding: 14px 18px; border-radius: 6px; margin-bottom: 14px;">
<b>📖 Connects to Slides 13–17</b><br>
This section builds the <b>Markov Decision Process (MDP)</b> that formalises our construction-site inspection task. The five components from Slide 13 are all present:
<ul style="margin: 8px 0 0 0;">
  <li><b>S</b> — 36 states (6×6 grid cells)</li>
  <li><b>A</b> — 4 actions: UP / DOWN / LEFT / RIGHT</li>
  <li><b>R</b> — reward function: +10 goal, −10 hazard, −1 step</li>
  <li><b>P</b> — deterministic transitions</li>
  <li><b>γ</b> — discount factor (set in section 2 / 4)</li>
</ul>
</div>

Running this section will display the **initial grid map** of the construction site.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Section 1 — CONSTRUCTION SITE GRID WORLD (MDP ENVIRONMENT)
#  Matches Slides 13–17: MDP definition + Grid World example
# ─────────────────────────────────────────────────────────────

class ConstructionGridWorld:
    """
    6×6 grid representing a construction floor-plan.

    MDP formalisation (Slide 13):
      S (States)      : (row, col) positions → 36 total
      A (Actions)     : {UP=0, DOWN=1, LEFT=2, RIGHT=3}
      R (Reward)      : +10 goal | −10 hazard | −1 per step
      P (Transition)  : deterministic next-state
      γ (Discount)    : passed in at training time
    """

    GRID = [
        ['S', '.', '.', 'H', '.', '.'],
        ['.', 'H', '.', '.', '.', '.'],
        ['.', '.', '.', 'H', '.', 'H'],
        ['.', '.', 'H', '.', '.', '.'],
        ['.', 'H', '.', '.', 'H', '.'],
        ['.', '.', '.', '.', '.', 'G'],
    ]

    ACTIONS     = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    ACTION_NAMES = {0: '↑ UP', 1: '↓ DOWN', 2: '← LEFT', 3: '→ RIGHT'}
    REWARDS     = {'G': 10.0, 'H': -10.0, '.': -1.0, 'S': -1.0}

    def __init__(self):
        self.nrows    = len(self.GRID)
        self.ncols    = len(self.GRID[0])
        self.n_states = self.nrows * self.ncols   # 36
        self.n_actions = 4
        self.start    = (0, 0)
        self._find_special_cells()
        self.reset()

    def _find_special_cells(self):
        self.hazards = []
        for r, row in enumerate(self.GRID):
            for c, cell in enumerate(row):
                if cell == 'G':
                    self.goal = (r, c)
                elif cell == 'H':
                    self.hazards.append((r, c))

    def reset(self):
        """Reset agent to start; return integer state id."""
        self.agent_pos = self.start
        return self._state_id(self.agent_pos)

    def _state_id(self, pos):
        """Convert (row, col) → integer state index."""
        return pos[0] * self.ncols + pos[1]

    def step(self, action):
        """
        Execute action, return (next_state, reward, done).
        Matches the MDP loop on Slide 14.
        """
        dr, dc = self.ACTIONS[action]
        nr = max(0, min(self.nrows - 1, self.agent_pos[0] + dr))
        nc = max(0, min(self.ncols - 1, self.agent_pos[1] + dc))
        self.agent_pos = (nr, nc)

        cell  = self.GRID[nr][nc]
        reward = self.REWARDS[cell]
        done   = cell in ('G', 'H')
        return self._state_id(self.agent_pos), reward, done

    # ── Visualisation ──────────────────────────────────────────
    def render(self, q_table=None, title='Construction Site Grid World',
               save_name=None):
        """Draw the grid; overlay policy arrows if q_table provided."""
        fig, ax = plt.subplots(figsize=(7, 7))
        colors = {'S': '#4A90D9', 'G': '#27AE60', 'H': '#E74C3C', '.': '#ECF0F1'}

        for r in range(self.nrows):
            for c in range(self.ncols):
                cell  = self.GRID[r][c]
                color = '#F39C12' if (r, c) == self.agent_pos else colors[cell]

                rect = plt.Rectangle([c, self.nrows - 1 - r], 1, 1,
                                      facecolor=color, edgecolor='white', lw=2)
                ax.add_patch(rect)

                label = {'S': 'START', 'G': 'GOAL', 'H': 'HAZARD', '.': ''}[cell]
                ax.text(c + 0.5, self.nrows - r - 0.5, label,
                        ha='center', va='center', fontsize=8, fontweight='bold',
                        color='white' if cell in ('H', 'G') else '#2C3E50')

                if q_table is not None and cell not in ('G', 'H'):
                    state  = r * self.ncols + c
                    best_a = int(np.argmax(q_table[state]))
                    dr_, dc_ = self.ACTIONS[best_a]
                    ax.annotate('',
                        xy=(c + 0.5 + dc_ * 0.3, self.nrows - r - 0.5 - dr_ * 0.3),
                        xytext=(c + 0.5, self.nrows - r - 0.5),
                        arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))

        ax.set_xlim(0, self.ncols)
        ax.set_ylim(0, self.nrows)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(title, fontsize=13, fontweight='bold', pad=12)

        patches = [mpatches.Patch(color=v, label=k)
                   for k, v in colors.items() if k != 'S']
        patches.insert(0, mpatches.Patch(color='#F39C12', label='Agent'))
        ax.legend(handles=patches, loc='upper right', fontsize=8,
                  bbox_to_anchor=(1.18, 1.0))
        plt.tight_layout()

        if save_name:
            path = save_path(save_name)
            plt.savefig(path, dpi=150, bbox_inches='tight')
            print(f'Figure saved → {path}')
        plt.show()


# ── Instantiate and display ────────────────────────────────────
env = ConstructionGridWorld()

print('MDP Summary')
print(f'  States   (S): {env.n_states}  ({env.nrows}×{env.ncols} grid)')
print(f'  Actions  (A): {env.n_actions}  {list(env.ACTION_NAMES.values())}')
print(f'  Rewards  (R): Goal=+10, Hazard=-10, Step=-1')
print(f'  Hazards     : {len(env.hazards)} cells')
print(f'  Goal        : row {env.goal[0]}, col {env.goal[1]}')
print()

env.render(title='Construction Site Grid World — Initial State',
           save_name='grid_world_initial.png')

---
## 📊 Section 2 — Tabular Q-Learning

<div style="background:#e8f5e9; border-left: 5px solid #27ae60; padding: 14px 18px; border-radius: 6px; margin-bottom: 14px;">
<b>📖 Connects to Slides 19–21</b><br>
Implements the classic <b>Q-table update rule</b> (Bellman equation, Slide 20):
</div>

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{TD target}} - Q(s,a)\Big]$$

**Key concepts demonstrated:**
- **ε-greedy** exploration/exploitation trade-off  
- Q-table as a lookup table `Q[state, action]`  
- Why this doesn't scale → motivation for DQN (Slide 21)

| Hyperparameter | Value | Meaning |
|---|---|---|
| α (alpha) | 0.1 | Learning rate |
| γ (gamma) | 0.95 | Discount factor |
| ε start | 1.0 | 100% explore at start |
| ε end | 0.05 | 5% explore after decay |
| Episodes | 1000 | Training iterations |

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Section 2 — TABULAR Q-LEARNING
#  Matches Slides 19–21: Value function, Bellman equation,
#  Value iteration, Q-table
#
#  ⚠️  Requires: Section 0 (imports) + Section 1 (env) to be run first
# ─────────────────────────────────────────────────────────────

class TabularQLearning:
    """
    Classic tabular Q-Learning.

    Core update (Bellman equation, Slide 20):
      Q(s,a) ← Q(s,a) + α · [r + γ·max_a' Q(s',a') − Q(s,a)]
                               ───────────────────────
                                      TD target

    Limitation (Slide 21): Must store Q(s,a) for ALL (s,a) pairs.
    With 36 states × 4 actions = 144 entries here — manageable.
    But for Atari (pixels) this becomes computationally infeasible!
    → Solution: use a neural network (see Section 4).
    """

    def __init__(self, env, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995):
        self.env           = env
        self.alpha         = alpha          # learning rate α
        self.gamma         = gamma          # discount factor γ
        self.epsilon       = epsilon        # exploration rate ε
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay

        # Q-table: (n_states × n_actions), initialised to zero
        self.Q = np.zeros((env.n_states, env.n_actions))

    # ── Action selection ──────────────────────────────────────
    def choose_action(self, state):
        """ε-greedy: explore randomly or exploit best known action."""
        if random.random() < self.epsilon:
            return random.randint(0, self.env.n_actions - 1)  # explore
        return int(np.argmax(self.Q[state]))                   # exploit

    # ── Bellman update ────────────────────────────────────────
    def update(self, state, action, reward, next_state, done):
        """
        One-step TD update (Bellman equation, Slide 20).
        If terminal (done=True) → no future reward, target = r only.
        """
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.Q[next_state])

        td_error = target - self.Q[state, action]
        self.Q[state, action] += self.alpha * td_error

    # ── Training loop ─────────────────────────────────────────
    def train(self, n_episodes=1000, max_steps=200, verbose=True):
        rewards_history, steps_history = [], []

        for ep in range(n_episodes):
            state        = self.env.reset()
            total_reward = 0

            for step in range(max_steps):
                action                      = self.choose_action(state)
                next_state, reward, done    = self.env.step(action)
                self.update(state, action, reward, next_state, done)
                state        = next_state
                total_reward += reward
                if done:
                    break

            # Decay ε after each episode
            self.epsilon = max(self.epsilon_min,
                               self.epsilon * self.epsilon_decay)
            rewards_history.append(total_reward)
            steps_history.append(step + 1)

            if verbose and (ep + 1) % 200 == 0:
                avg = np.mean(rewards_history[-200:])
                print(f'  Episode {ep+1:>5}/{n_episodes}  '
                      f'Avg Reward (last 200): {avg:+6.1f}  '
                      f'ε = {self.epsilon:.3f}')

        return rewards_history, steps_history

    # ── Q-table printer ───────────────────────────────────────
    def print_q_table(self):
        """Print human-readable Q-table with best action highlighted."""
        print('\n── Learned Q-Table ──────────────────────────────────────────')
        header = f"{'State':>8}  " + '  '.join(
            f'{n:>10}' for n in self.env.ACTION_NAMES.values())
        print(header)
        print('─' * (len(header) + 2))
        for s in range(self.env.n_states):
            r, c  = divmod(s, self.env.ncols)
            cell  = self.env.GRID[r][c]
            vals  = '  '.join(f'{self.Q[s, a]:>10.3f}'
                               for a in range(self.env.n_actions))
            row   = f'  ({r},{c}) {cell}  {vals}'
            if cell not in ('G', 'H'):
                best = int(np.argmax(self.Q[s]))
                row += f'   ← best: {self.env.ACTION_NAMES[best]}'
            print(row)


# ── Train ─────────────────────────────────────────────────────
print('=' * 60)
print('  PART 2 — Tabular Q-Learning  (Slides 19–21)')
print('=' * 60)
print('Hyperparameters:')
print('  α (learning rate)  = 0.1')
print('  γ (discount)       = 0.95')
print('  ε: 1.0 → 0.05 (ε-decay = 0.995)')
print('  Episodes           = 1,000')
print()

agent_tab = TabularQLearning(env, alpha=0.1, gamma=0.95,
                              epsilon=1.0, epsilon_min=0.05,
                              epsilon_decay=0.995)
rewards_tab, steps_tab = agent_tab.train(n_episodes=1000, verbose=True)
agent_tab.print_q_table()
print('\n✅  Q-Learning training complete. Run Cell 3 to visualise.')

---
## 📈 Section 3 — Tabular Q-Learning: Visualisation

<div style="background:#fce4ec; border-left: 5px solid #e74c3c; padding: 14px 18px; border-radius: 6px; margin-bottom: 14px;">
<b>📖 Connects to Slide 31</b><br>
Produces three visualisations showing what the agent has learned:
<ol style="margin: 8px 0 0 0;">
  <li><b>Training reward curve</b> — shows convergence over episodes</li>
  <li><b>Q-value heatmap</b> — shows learned value for action → RIGHT</li>
  <li><b>Policy map</b> — overlays the greedy policy arrows on the grid</li>
</ol>
</div>

> 📌 **Interpretation tip:** By the end of training, the reward curve should stabilise near **0 or above**, meaning the agent consistently reaches the goal without too many wasted steps.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Section 3 — TABULAR Q-LEARNING: VISUALISATION
#  Training curve + Q-value heatmap + learned policy map
#
#  ⚠️  Requires: Section 0 + Section 1 + Section 2 to be run first
# ─────────────────────────────────────────────────────────────

def plot_tabular_results(agent, rewards, steps, window=50):
    """
    Figure 1: Learning curve (reward & steps per episode).
    Figure 2: Q-value heatmap for the RIGHT action.
    """
    # ── Figure 1: Learning curves ──────────────────────────────
    smoothed_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smoothed_s = np.convolve(steps,   np.ones(window)/window, mode='valid')
    x_smooth   = range(window - 1, len(rewards))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Tabular Q-Learning — Training Progress  (Slide 31)',
                 fontsize=13, fontweight='bold', y=1.02)

    # Reward
    axes[0].plot(rewards, alpha=0.25, color='steelblue', label='Per-episode')
    axes[0].plot(x_smooth, smoothed_r, color='steelblue', lw=2.5,
                 label=f'Moving avg (n={window})')
    axes[0].axhline(0, color='gray', linestyle='--', lw=0.8, label='Zero reward')
    axes[0].set_xlabel('Episode',      fontsize=11)
    axes[0].set_ylabel('Total Reward', fontsize=11)
    axes[0].set_title('Reward per Episode')
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # Steps
    axes[1].plot(steps, alpha=0.25, color='darkorange', label='Per-episode')
    axes[1].plot(x_smooth, smoothed_s, color='darkorange', lw=2.5,
                 label=f'Moving avg (n={window})')
    axes[1].set_xlabel('Episode',           fontsize=11)
    axes[1].set_ylabel('Steps to Finish',   fontsize=11)
    axes[1].set_title('Steps per Episode (lower = more efficient)')
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path('q_learning_training.png'), dpi=150, bbox_inches='tight')
    print(f'Figure saved → {save_path("q_learning_training.png")}')
    plt.show()

    # ── Figure 2: Q-value heatmap ──────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle('Q-Value Heatmaps — One Panel per Action',
                 fontsize=13, fontweight='bold')

    for a, name in agent.env.ACTION_NAMES.items():
        q_map = agent.Q[:, a].reshape(agent.env.nrows, agent.env.ncols)
        im    = axes[a].imshow(q_map, cmap='RdYlGn', aspect='auto',
                               vmin=agent.Q.min(), vmax=agent.Q.max())
        plt.colorbar(im, ax=axes[a], fraction=0.046)
        axes[a].set_title(f'Q(s, {name})', fontsize=10)
        axes[a].set_xlabel('Column')
        axes[a].set_ylabel('Row')

    plt.tight_layout()
    plt.savefig(save_path('q_value_heatmaps.png'), dpi=150, bbox_inches='tight')
    print(f'Figure saved → {save_path("q_value_heatmaps.png")}')
    plt.show()


# ── Run visualisation ─────────────────────────────────────────
plot_tabular_results(agent_tab, rewards_tab, steps_tab)

# ── Policy map on grid ────────────────────────────────────────
env.render(q_table=agent_tab.Q,
           title='Learned Policy — Tabular Q-Learning\n(arrows = greedy action per cell)',
           save_name='grid_world_policy_tabular.png')

print('\n📊  Final Q-table statistics:')
print(f'   Q max  = {agent_tab.Q.max():.3f}  (highest-value state-action)')
print(f'   Q min  = {agent_tab.Q.min():.3f}  (lowest-value state-action)')
print(f'   Q mean = {agent_tab.Q[agent_tab.Q != 0].mean():.3f}  (non-zero entries)')

---
## 🧠 Section 4 — Deep Q-Network (DQN)

<div style="background:#e8eaf6; border-left: 5px solid #5c6bc0; padding: 14px 18px; border-radius: 6px; margin-bottom: 14px;">
<b>📖 Connects to Slides 22–31</b><br>
Replaces the Q-table with a <b>neural network</b> Q(s,a;θ) — the core idea of <b>Deep Q-Learning</b>.
Three key innovations from the DQN paper (Mnih et al. 2015):
<ol style="margin: 8px 0 0 0;">
  <li><b>Function approximator</b> — neural net approximates Q(s,a) for all actions simultaneously (Slide 22–29)</li>
  <li><b>Experience Replay</b> — break temporal correlations by sampling random mini-batches from a replay buffer (Slide 30)</li>
  <li><b>Target Network θ⁻</b> — a separate frozen copy for computing TD targets, updated every C steps (Slide 31)</li>
</ol>
</div>

**Loss function (Slide 23):**
$$\mathcal{L}(\theta) = \mathbb{E}\Big[\Big(\underbrace{r + \gamma \max_{a'} Q(s',a';\theta^-)}_\text{TD target (frozen)} - Q(s,a;\theta)\Big)^2\Big]$$

**Network architecture:**
```
Input: one-hot(state)  [36-dim]
    → Linear(36→64) → ReLU
    → Linear(64→64) → ReLU
    → Linear(64→4)          ← Q(s, UP/DOWN/LEFT/RIGHT)
```

> ⚠️ **Requires PyTorch.** If Section 0 printed a warning about PyTorch, run `pip install torch` first.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Section 4 — DEEP Q-NETWORK (DQN)
#  Matches Slides 22–31: Neural network Q-function,
#  Experience Replay buffer, Target network, DQN training loop
#
#  ⚠️  Requires: Section 0 + Section 1 to be run first
#  ⚠️  Requires PyTorch (pip install torch)
# ─────────────────────────────────────────────────────────────

if not TORCH_AVAILABLE:
    print('❌  PyTorch not available.')
    print('   Install it with:  pip install torch')
    print('   Then restart the kernel and re-run from Section 0.')
else:

    # ══════════════════════════════════════════════════════════
    #  1. Q-NETWORK  (the function approximator — Slides 22–29)
    # ══════════════════════════════════════════════════════════
    class QNetwork(nn.Module):
        """
        Neural network approximating Q(s, a; θ).

        Input  : one-hot encoded state  (size = n_states = 36)
        Hidden : two FC layers with ReLU activations
        Output : Q-value for every action simultaneously (size = 4)

        This is the 'function approximator' from Slide 22.
        For Atari the input would be conv layers over pixel frames;
        here we use one-hot encoding for simplicity.
        """
        def __init__(self, n_states, n_actions, hidden=64):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_states, hidden),  # Input layer
                nn.ReLU(),
                nn.Linear(hidden, hidden),    # Hidden layer
                nn.ReLU(),
                nn.Linear(hidden, n_actions)  # Output layer: one Q-value per action
            )

        def forward(self, x):
            return self.net(x)

    # ══════════════════════════════════════════════════════════
    #  2. EXPERIENCE REPLAY BUFFER  (Slide 30)
    # ══════════════════════════════════════════════════════════
    class ReplayBuffer:
        """
        Circular buffer storing (s, a, r, s', done) transitions.

        Why this matters (Slide 30):
          Problem:  consecutive samples are strongly correlated
                    → biased gradient updates → poor learning
          Solution: randomly sample a mini-batch from the buffer
                    → breaks correlations → stable training
        """
        def __init__(self, capacity=10_000):
            self.buffer = deque(maxlen=capacity)

        def push(self, state, action, reward, next_state, done):
            self.buffer.append((state, action, reward, next_state, done))

        def sample(self, batch_size):
            batch                               = random.sample(self.buffer, batch_size)
            states, actions, rewards, nstates, dones = zip(*batch)
            return (np.array(states),
                    np.array(actions),
                    np.array(rewards,  dtype=np.float32),
                    np.array(nstates),
                    np.array(dones,    dtype=np.float32))

        def __len__(self):
            return len(self.buffer)

    # ══════════════════════════════════════════════════════════
    #  3. DQN AGENT  (full algorithm, Slides 22–31)
    # ══════════════════════════════════════════════════════════
    class DQNAgent:
        """
        DQN with experience replay and target network.
        Implements Algorithm 1 from Mnih et al. 2015 (Slide 31).

        Two networks:
          q_net      — online net, updated every step  (weights θ)
          target_net — frozen copy, synced every C steps (weights θ⁻)
        """
        def __init__(self, env, lr=1e-3, gamma=0.95,
                     epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.997,
                     batch_size=64, target_update_freq=100,
                     buffer_capacity=10_000, hidden=64):

            self.env               = env
            self.gamma             = gamma
            self.epsilon           = epsilon
            self.epsilon_min       = epsilon_min
            self.epsilon_decay     = epsilon_decay
            self.batch_size        = batch_size
            self.target_update_freq = target_update_freq
            self.n_states          = env.n_states
            self.n_actions         = env.n_actions

            # Build online and target networks
            self.q_net      = QNetwork(self.n_states, self.n_actions, hidden)
            self.target_net = QNetwork(self.n_states, self.n_actions, hidden)
            self.target_net.load_state_dict(self.q_net.state_dict())
            self.target_net.eval()   # target net is never trained directly

            self.optimizer  = optim.Adam(self.q_net.parameters(), lr=lr)
            self.loss_fn    = nn.MSELoss()   # L(θ) from Slide 23
            self.memory     = ReplayBuffer(buffer_capacity)
            self.steps_done = 0

        def _encode(self, state_id):
            """One-hot encode integer state id → float32 numpy vector."""
            v = np.zeros(self.n_states, dtype=np.float32)
            v[state_id] = 1.0
            return v

        def choose_action(self, state):
            """ε-greedy: explore or exploit."""
            if random.random() < self.epsilon:
                return random.randint(0, self.n_actions - 1)
            with torch.no_grad():
                s = torch.FloatTensor(self._encode(state)).unsqueeze(0)
                return int(self.q_net(s).argmax().item())

        def _update_weights(self):
            """
            Mini-batch Bellman loss + backprop (Slide 23–24).

            y_i = r  +  γ · max_a' Q(s', a'; θ⁻)   [using TARGET net]
            L   = MSE( Q(s, a; θ), y_i )             [update ONLINE net]
            """
            if len(self.memory) < self.batch_size:
                return None

            states, actions, rewards, next_states, dones = \
                self.memory.sample(self.batch_size)

            # Encode to tensors
            s_t    = torch.FloatTensor(np.array([self._encode(s) for s in states]))
            s_next = torch.FloatTensor(np.array([self._encode(s) for s in next_states]))
            a_t    = torch.LongTensor(actions)
            r_t    = torch.FloatTensor(rewards)
            done_t = torch.FloatTensor(dones)

            # Forward pass: Q(s, a; θ)  — online net
            q_current = self.q_net(s_t).gather(1, a_t.unsqueeze(1)).squeeze(1)

            # TD target: y = r + γ·max Q(s',a';θ⁻)·(1−done) — target net
            with torch.no_grad():
                q_next_max = self.target_net(s_next).max(1)[0]
                y = r_t + self.gamma * q_next_max * (1 - done_t)

            # Loss + backward pass (gradient update — Slide 24)
            loss = self.loss_fn(q_current, y)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
            self.optimizer.step()
            return loss.item()

        def train(self, n_episodes=1000, max_steps=200, verbose=True):
            """Full DQN training loop — Algorithm 1 from Mnih et al."""
            rewards_history, loss_history = [], []

            for ep in range(n_episodes):
                state        = self.env.reset()
                total_reward = 0
                ep_losses    = []

                for step in range(max_steps):
                    action                   = self.choose_action(state)
                    next_state, reward, done = self.env.step(action)

                    # Store transition in replay buffer (Slide 36)
                    self.memory.push(state, action, reward, next_state, done)

                    # Gradient step on Bellman loss (Slide 37)
                    loss = self._update_weights()
                    if loss is not None:
                        ep_losses.append(loss)

                    state        = next_state
                    total_reward += reward
                    self.steps_done += 1

                    # Sync target network every C steps
                    if self.steps_done % self.target_update_freq == 0:
                        self.target_net.load_state_dict(self.q_net.state_dict())

                    if done:
                        break

                self.epsilon = max(self.epsilon_min,
                                   self.epsilon * self.epsilon_decay)
                rewards_history.append(total_reward)
                if ep_losses:
                    loss_history.append(np.mean(ep_losses))

                if verbose and (ep + 1) % 200 == 0:
                    avg_r = np.mean(rewards_history[-200:])
                    avg_l = np.mean(loss_history[-200:]) if loss_history else 0
                    print(f'  Episode {ep+1:>5}/{n_episodes}  '
                          f'Avg Reward: {avg_r:+6.1f}  '
                          f'Loss: {avg_l:.4f}  '
                          f'ε = {self.epsilon:.3f}')

            return rewards_history, loss_history

        def get_q_table(self):
            """Extract Q-values from network for all states (for policy vis)."""
            q = np.zeros((self.env.n_states, self.env.n_actions))
            with torch.no_grad():
                for s in range(self.env.n_states):
                    enc  = torch.FloatTensor(self._encode(s)).unsqueeze(0)
                    q[s] = self.q_net(enc).numpy()
            return q


    # ── Train ─────────────────────────────────────────────────
    print('=' * 60)
    print('  PART 3 — Deep Q-Network (DQN)  (Slides 22–31)')
    print('=' * 60)
    print('Architecture:')
    print('  Input  : one-hot state  [36-dim]')
    print('  Hidden : Linear(36→64) → ReLU → Linear(64→64) → ReLU')
    print('  Output : Linear(64→4)  [Q-value for each action]')
    print('Hyperparameters:')
    print('  lr (Adam)          = 1e-3')
    print('  γ (discount)       = 0.95')
    print('  Replay buffer size = 10,000')
    print('  Batch size         = 64')
    print('  Target net update  = every 100 steps')
    print('  Episodes           = 1,000')
    print()

    env.reset()
    dqn_agent = DQNAgent(env, lr=1e-3, gamma=0.95,
                         epsilon=1.0, epsilon_min=0.05,
                         epsilon_decay=0.997, batch_size=64,
                         target_update_freq=100)
    rewards_dqn, losses_dqn = dqn_agent.train(n_episodes=1000, verbose=True)
    print('\n✅  DQN training complete. Run Section 5 to visualise.')

---
## 📈 Section 5 — DQN: Visualisation

<div style="background:#e0f2f1; border-left: 5px solid #00897b; padding: 14px 18px; border-radius: 6px; margin-bottom: 14px;">
<b>📖 Connects to Slides 29–31</b><br>
Visualises DQN training with three plots:
<ol style="margin: 8px 0 0 0;">
  <li><b>Reward curve</b> — convergence of the DQN agent</li>
  <li><b>Bellman loss L(θ)</b> — MSE loss decreasing as the Q-network improves</li>
  <li><b>Policy map</b> — DQN greedy policy overlaid on the construction grid</li>
</ol>
</div>

> 📌 **Compare with Section 3**: DQN uses more parameters and more machinery (replay buffer, target network), but can scale to **pixel inputs** and **continuous state spaces** — unlike the Q-table in Section 2.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Section 5 — DQN: VISUALISATION
#  Reward curve + Bellman loss + learned policy map
#
#  ⚠️  Requires: Section 0 + Section 1 + Section 4 to be run first
# ─────────────────────────────────────────────────────────────

if not TORCH_AVAILABLE:
    print('❌  Skipped — PyTorch not available (see Section 4).')
else:

    def plot_dqn_results(rewards, losses, window=50):
        """Plot reward curve and Bellman loss side-by-side."""
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        fig.suptitle('Deep Q-Network (DQN) — Training Progress  (Slide 31)',
                     fontsize=13, fontweight='bold', y=1.02)

        # ── Reward ──────────────────────────────────────────────
        smoothed_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0].plot(rewards, alpha=0.25, color='royalblue', label='Per-episode')
        axes[0].plot(range(window - 1, len(rewards)), smoothed_r,
                     color='royalblue', lw=2.5,
                     label=f'Moving avg (n={window})')
        axes[0].axhline(0, color='gray', linestyle='--', lw=0.8)
        axes[0].set_xlabel('Episode',      fontsize=11)
        axes[0].set_ylabel('Total Reward', fontsize=11)
        axes[0].set_title('DQN Reward per Episode')
        axes[0].legend(fontsize=9)
        axes[0].grid(alpha=0.3)

        # ── Bellman Loss ─────────────────────────────────────────
        if losses:
            w2         = min(window, len(losses))
            smoothed_l = np.convolve(losses, np.ones(w2)/w2, mode='valid')
            axes[1].plot(losses, alpha=0.25, color='tomato', label='Per-episode')
            axes[1].plot(range(w2 - 1, len(losses)), smoothed_l,
                         color='tomato', lw=2.5,
                         label=f'Moving avg (n={w2})')
            axes[1].set_xlabel('Episode',  fontsize=11)
            axes[1].set_ylabel('MSE Loss', fontsize=11)
            axes[1].set_title('Bellman Loss  L(θ) = E[(y − Q(s,a;θ))²]')
            axes[1].legend(fontsize=9)
            axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(save_path('dqn_training.png'), dpi=150, bbox_inches='tight')
        print(f'Figure saved → {save_path("dqn_training.png")}')
        plt.show()


    # ── Plot training curves ───────────────────────────────────
    plot_dqn_results(rewards_dqn, losses_dqn)

    # ── Policy map on grid ────────────────────────────────────
    q_from_net = dqn_agent.get_q_table()
    env.render(q_table=q_from_net,
               title='Learned Policy — Deep Q-Network (DQN)\n(arrows = greedy action per cell)',
               save_name='grid_world_policy_dqn.png')

    # ── Side-by-side policy comparison ───────────────────────
    print('\n── Tabular Q-Learning vs DQN: Final Rewards ────────────')
    print(f'  Tabular Q (last 100 ep avg): {np.mean(rewards_tab[-100:]):+.2f}')
    print(f'  DQN       (last 100 ep avg): {np.mean(rewards_dqn[-100:]):+.2f}')
    print()
    print('📁  All output files saved to:', OUTPUT_DIR)
    print('    • grid_world_initial.png')
    print('    • q_learning_training.png')
    print('    • q_value_heatmaps.png')
    print('    • grid_world_policy_tabular.png')
    print('    • dqn_training.png')
    print('    • grid_world_policy_dqn.png')